# Task 9: Docker Fundamentals and Container Lifecycle Management
**Objective:** understand Docker architecture and manage containerized deep learning applications.

**Before running:** start Docker Desktop and wait until it shows *Engine running*. Run the cells top to bottom (Kernel > Restart & Run All). Each cell executes real Docker CLI commands and prints their output.

In [ ]:
import task9_docker as t
assert t.docker_ready(), 'Docker is not running - start Docker Desktop first'
t.write_build_context()
results = []
print(open('dl_app/Dockerfile').read())
print(open('dl_app/app.py').read())

## 1. Install Docker and verify the setup
Docker uses a client-server architecture: the Docker CLI (client) sends REST API calls to the Docker daemon (dockerd), which manages images, containers, networks and volumes and delegates container execution to containerd and runc. Docker Desktop bundles the engine, CLI, a Linux VM (on Windows/macOS), and a GUI.

**Step 1: Docker client/server version**

`docker --version && docker version --format 'Client: {{.Client.Version}} | Server: {{.Server.Version}}'`

*Observation:* The client (CLI) talks to the Docker daemon (server) over a socket. Both versions confirm the installation works.

In [ ]:
r = t.run_step(0, results)

**Step 2: Engine information**

`docker info --format 'Containers: {{.Containers}} | Images: {{.Images}} | Storage driver: {{.Driver}} | OS: {{.OperatingSystem}}'`

*Observation:* Shows the daemon's state, the storage driver (overlay2 is typical) and the host OS the engine runs on.

In [ ]:
r = t.run_step(1, results)

**Step 3: Run the hello-world test container**

`docker run --rm hello-world`

*Observation:* The daemon did not have the image locally, so it pulled it from Docker Hub, created a container, ran it, and streamed the output back to the CLI.

In [ ]:
r = t.run_step(2, results)

## 2. Create and manage containers
A container is a running instance of an image: an isolated process using Linux namespaces (isolation) and cgroups (resource limits), with a thin writable layer on top of read-only image layers.

**Step 4: Run a detached web container with port mapping**

`docker run -d --name dl-web -p 8080:80 nginx:alpine`

*Observation:* -d runs in the background, --name gives a friendly name, -p maps host port 8080 to container port 80.

In [ ]:
r = t.run_step(3, results)

**Step 5: List running containers**

`docker ps`

*Observation:* Shows container ID, image, command, status, port mapping and name of running containers.

In [ ]:
r = t.run_step(4, results)

**Step 6: Call the container from the host**

`curl -s -o /dev/null -w 'HTTP status: %{http_code}\n' http://localhost:8080`

*Observation:* HTTP 200 proves the published port forwards host traffic into the container.

In [ ]:
r = t.run_step(5, results)

**Step 7: Run a command inside the running container**

`docker exec dl-web sh -c 'echo Hello from inside the container && nginx -v && hostname'`

*Observation:* docker exec starts an extra process inside an existing container; the hostname equals the container ID.

In [ ]:
r = t.run_step(6, results)

**Step 8: View container logs**

`docker logs --tail 5 dl-web`

*Observation:* Logs capture the container's stdout/stderr (nginx access logs here).

In [ ]:
r = t.run_step(7, results)

**Step 9: Resource usage snapshot**

`docker stats --no-stream dl-web`

*Observation:* Shows live CPU, memory, network and block I/O of the container.

In [ ]:
r = t.run_step(8, results)

**Step 10: Inspect container metadata**

`docker inspect --format 'Name={{.Name}} State={{.State.Status}} IP={{.NetworkSettings.IPAddress}} Image={{.Config.Image}}' dl-web`

*Observation:* docker inspect returns the full JSON configuration; --format extracts specific fields.

In [ ]:
r = t.run_step(9, results)

## 3. Explore Docker images and registries
An image is an immutable, layered template. A registry (Docker Hub, GHCR, ECR, private registries) stores and distributes images by repository:tag and digest. A Dockerfile describes how to build an image reproducibly, which is critical for deep learning environments with pinned library versions.

**Step 11: Pull an image from the registry (Docker Hub)**

`docker pull python:3.11-slim`

*Observation:* Images are made of read-only layers identified by digests; layers already present are not downloaded again.

In [ ]:
r = t.run_step(10, results)

**Step 12: Search the registry**

`docker search tensorflow --limit 5`

*Observation:* Docker Hub is the default public registry; search lists matching repositories with stars and official flags.

In [ ]:
r = t.run_step(11, results)

**Step 13: List local images**

`docker images`

*Observation:* Shows repository, tag, image ID, creation date and size of each local image.

In [ ]:
r = t.run_step(12, results)

**Step 14: Image layers (history)**

`docker history python:3.11-slim`

*Observation:* Every Dockerfile instruction creates a layer; this lists them with their sizes.

In [ ]:
r = t.run_step(13, results)

**Step 15: Build a custom DL-style image from a Dockerfile**

`docker build -t dl-app:1.0 ./dl_app`

*Observation:* docker build reads the Dockerfile, executes each instruction as a cached layer, and tags the result dl-app:1.0.

In [ ]:
r = t.run_step(14, results)

**Step 16: Run the custom image**

`docker run --rm dl-app:1.0`

*Observation:* Runs the containerized NumPy neural-network inference; the container exits when the process finishes.

In [ ]:
r = t.run_step(15, results)

**Step 17: Tag the image for a registry**

`docker tag dl-app:1.0 yourdockerhubuser/dl-app:1.0 && docker images yourdockerhubuser/dl-app`

*Observation:* Tagging prepares the image to be pushed with 'docker push' to a registry (replace yourdockerhubuser with your account).

In [ ]:
r = t.run_step(16, results)

## 4. Container lifecycle operations
Container states: Created -> Running -> Paused -> Exited -> Removed. Transitions are driven by create, start, pause/unpause, stop/kill, restart and rm.

**Step 18: Create a container without starting it**

`docker create --name lifecycle-demo python:3.11-slim sleep 600 && docker ps -a --filter name=lifecycle-demo --format 'table {{.Names}}\t{{.Status}}'`

*Observation:* State: Created. The writable layer exists but no process is running.

In [ ]:
r = t.run_step(17, results)

**Step 19: Start**

`docker start lifecycle-demo && docker ps --filter name=lifecycle-demo --format 'table {{.Names}}\t{{.Status}}'`

*Observation:* State: Running.

In [ ]:
r = t.run_step(18, results)

**Step 20: Pause**

`docker pause lifecycle-demo && docker ps --filter name=lifecycle-demo --format 'table {{.Names}}\t{{.Status}}'`

*Observation:* State: Paused. Processes are frozen using the cgroup freezer (memory is kept).

In [ ]:
r = t.run_step(19, results)

**Step 21: Unpause**

`docker unpause lifecycle-demo && docker ps --filter name=lifecycle-demo --format 'table {{.Names}}\t{{.Status}}'`

*Observation:* Processes resume exactly where they stopped.

In [ ]:
r = t.run_step(20, results)

**Step 22: Stop (graceful, SIGTERM then SIGKILL)**

`docker stop -t 2 lifecycle-demo && docker ps -a --filter name=lifecycle-demo --format 'table {{.Names}}\t{{.Status}}'`

*Observation:* State: Exited. Docker sends SIGTERM, waits, then SIGKILL; the filesystem is kept.

In [ ]:
r = t.run_step(21, results)

**Step 23: Restart**

`docker restart lifecycle-demo && docker ps --filter name=lifecycle-demo --format 'table {{.Names}}\t{{.Status}}'`

*Observation:* Stop followed by start; the container returns to Running.

In [ ]:
r = t.run_step(22, results)

**Step 24: Kill (immediate SIGKILL)**

`docker kill lifecycle-demo && docker ps -a --filter name=lifecycle-demo --format 'table {{.Names}}\t{{.Status}}'`

*Observation:* Forces immediate termination, exit code 137 (128 + 9).

In [ ]:
r = t.run_step(23, results)

**Step 25: Restart policy demo**

`docker run -d --name policy-demo --restart unless-stopped nginx:alpine && docker inspect --format 'RestartPolicy={{.HostConfig.RestartPolicy.Name}}' policy-demo`

*Observation:* Restart policies (no, on-failure, always, unless-stopped) let the daemon restart containers automatically.

In [ ]:
r = t.run_step(24, results)

**Step 26: Remove containers**

`docker rm -f lifecycle-demo policy-demo && docker ps -a --format 'table {{.Names}}\t{{.Status}}'`

*Observation:* docker rm deletes the container and its writable layer; the image remains.

In [ ]:
r = t.run_step(25, results)

## 5. Networking concepts
Docker network drivers: bridge (default, single host), host (shares host network stack), none (no networking), overlay (multi-host swarm). User-defined bridge networks add DNS resolution by container name and better isolation.

**Step 27: List default networks**

`docker network ls`

*Observation:* Default networks: bridge (default for containers), host and none.

In [ ]:
r = t.run_step(26, results)

**Step 28: Create a user-defined bridge network**

`docker network create dl-net`

*Observation:* User-defined bridges provide automatic DNS-based service discovery between containers.

In [ ]:
r = t.run_step(27, results)

**Step 29: Start two containers on the network**

`docker run -d --name backend --network dl-net nginx:alpine && docker run -d --name client --network dl-net alpine sleep 300`

*Observation:* Both containers join dl-net and get private IP addresses.

In [ ]:
r = t.run_step(28, results)

**Step 30: Container-to-container communication by name**

`docker exec client sh -c 'ping -c 2 backend && wget -qO- http://backend | head -n 4'`

*Observation:* The name 'backend' resolves through Docker's embedded DNS, proving container-to-container networking.

In [ ]:
r = t.run_step(29, results)

**Step 31: Inspect the network**

`docker network inspect dl-net --format '{{range .Containers}}{{.Name}} -> {{.IPv4Address}}{{println}}{{end}}'`

*Observation:* Lists the containers attached to the network with their IP addresses.

In [ ]:
r = t.run_step(30, results)

## 6. Storage concepts (volumes and bind mounts)
Container filesystems are ephemeral. Volumes (Docker-managed) and bind mounts (host paths) persist data beyond the container lifecycle. tmpfs mounts keep data in memory only.

**Step 32: Create a named volume**

`docker volume create dl-data && docker volume ls --filter name=dl-data`

*Observation:* Volumes are managed by Docker and live outside the container's writable layer.

In [ ]:
r = t.run_step(31, results)

**Step 33: Write data from one container**

`docker run --rm -v dl-data:/data alpine sh -c "echo 'model checkpoint v1' > /data/checkpoint.txt && ls -l /data"`

*Observation:* The container writes into the volume and is then deleted.

In [ ]:
r = t.run_step(32, results)

**Step 34: Read the data from a NEW container (persistence)**

`docker run --rm -v dl-data:/data alpine cat /data/checkpoint.txt`

*Observation:* The data survived container deletion, proving volume persistence (key for datasets and trained models).

In [ ]:
r = t.run_step(33, results)

**Step 35: Bind mount a host folder**

`mkdir -p host_data && echo 'dataset sample from host' > host_data/sample.txt && docker run --rm -v "$(pwd)/host_data:/mnt/host" alpine cat /mnt/host/sample.txt`

*Observation:* A bind mount maps a host directory into the container; changes are visible on both sides.

In [ ]:
r = t.run_step(34, results)

**Step 36: Inspect the volume**

`docker volume inspect dl-data --format 'Name={{.Name}} Driver={{.Driver}} Mountpoint={{.Mountpoint}}'`

*Observation:* Shows where Docker stores the volume on the host.

In [ ]:
r = t.run_step(35, results)

## 7. Cleanup


**Step 37: Remove all demo resources**

`docker rm -f dl-web backend client 2>&1; docker network rm dl-net; docker volume rm dl-data; docker ps -a --format 'table {{.Names}}\t{{.Status}}'`

*Observation:* Cleaning up avoids leftover containers, networks and volumes.

In [ ]:
r = t.run_step(36, results)

## Save the log and build the PDF report
Add Docker Desktop screenshots to the `screenshots/` folder first (Containers, Images, Volumes, Networks tabs).

In [ ]:
t.save_log(results)
t.build_pdf(results)

## Conclusion
Docker packages applications with their dependencies into portable images, runs them as isolated containers with a well-defined lifecycle, connects them through networks, and persists data with volumes - which makes deep learning environments reproducible across machines.